# Pocket-Agent Hackathon Pipeline (Colab Extension)

This notebook connects to your Colab GPU instance and automatically pulls your code from GitHub so that all the `make` commands work seamlessly.

In [1]:
# Step 1: Pull your code from GitHub into the Colab server
!git clone https://github.com/MAbdullahWaqar/pocket-agent-1.git pocket-agent-cloud
%cd pocket-agent-cloud/
print("✅ Code successfully downloaded from GitHub!")

fatal: destination path 'pocket-agent-cloud' already exists and is not an empty directory.
/content/pocket-agent-cloud
✅ Code successfully downloaded from GitHub!


In [2]:
# Step 2: Install requirements
!make install

pip install -r requirements.txt


In [3]:
# Step 3: Generate the synthetic dataset
!make data

python data/generate_data.py
Generating data...
Generated 1500 examples. Saved to /content/pocket-agent-cloud/data/train.jsonl.


In [15]:
!git pull


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 496 bytes | 496.00 KiB/s, done.
From https://github.com/MAbdullahWaqar/pocket-agent-1
   4356217..413f1ce  main       -> origin/main
Updating 4356217..413f1ce
Fast-forward
 starter/public_test.jsonl | 144 ++++++----------------------------------------
 1 file changed, 16 insertions(+), 128 deletions(-)


In [5]:
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (664.8 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cufft_cu1

In [8]:
# Step 4: Fine-tune the Qwen model via QLoRA
!make train

python train/finetune.py
INFO:__main__:Loading dataset from /content/pocket-agent-cloud/train/../data/train.jsonl
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"
INFO:__main__:Initializing Tokenizer...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/

In [9]:
!make quantize

python quantize/quantize.py
INFO:__main__:Loading base model to merge adapter...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
Loading weights: 100% 290/290 [00:00<00:00, 4638.60it/s]
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen

In [16]:
# Step 6: Run Evaluation against test set
!make eval

python eval/evaluate.py
STARTING EVALUATION
INFO:inference:Loading GGUF model from /content/pocket-agent-cloud/artifacts/model.Q4_K_M.gguf
llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
Prompt: What is the weather like in Chicago in F?
Expected: <tool_call>{"tool": "weather", "args": {"location": "Chicago", "unit": "F"}}</tool_call>
Predicted: <tool_call>{"tool": "weather", "args": {"city": "Chicago", "unit": "F"}}</tool_call>
Score: 0.5 | Latency: 8669.53 ms
--------------------------------------------------
Prompt: Convert 100 miles to km
Expected: <tool_call>{"tool": "convert", "args": {"value": 100.0, "from_unit": "miles", "to_unit": "km"}}</tool_call>
Predicted: <tool_call>{"tool": "convert", "args": {"distance": 100, "from": "miles", "to": "km"}}</tool_call>
Score: 0.5 | Latency: 2379.82 ms
--------------------------------------------------
Prompt: How much is 500 USD in EUR?
Expected: <tool_call>{"tool": "currency", "

### Step 7: Run Streamlit Demo
To view the Streamlit UI, we will use localtunnel to expose the port from the Colab kernel to your browser.

In [ ]:
import urllib
print("Password/Endpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

!streamlit run demo.py & npx -y localtunnel --port 8501
